# 06 – DistilBERT Model Fine-Tuning

**Purpose:** Train and fine-tune a `DistilBERT` sequence classifier for ticketing routing on Banking77.

This notebook demonstrates:
1. Centralized configuration lookup.
2. Fine-tuning utilizing mixed precision (AMP) and gradient accumulation.
3. Checkpoint resilience and early stopping monitoring.
4. Loading and plotting loss/accuracy learning curves from the training run.

## 0. Setup and Environment

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")

## 1. Execute Smoke Run Training

We execute a quick smoke run fine-tuning pipeline to check training and metrics exports on a sub-slice of samples.

In [ ]:
from src.models.transformer.train import train_model

test_metrics = train_model(
    smoke_run=True,
    resume=False
)
print("Smoke training complete!")
print("Final test metrics:")
print(test_metrics)

## 2. Learning Curves Visualization

We load the generated `training_history.csv` file and plot the training loss vs validation loss curves.

In [ ]:
from src.utils.constants import OUTPUT_DIR

history_path = OUTPUT_DIR / "metrics" / "training_history.csv"
history_df = pd.read_csv(history_path)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Losses
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="Train Loss", marker='o')
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="Val Loss", marker='s')
axes[0].set_title("Training and Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

# Plot Accuracy
axes[1].plot(history_df["epoch"], history_df["val_accuracy"] * 100, label="Val Accuracy", color='g', marker='^')
axes[1].set_title("Validation Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Export Phase Manifest
import json
from src.utils.artifacts import save_yaml
from src.api.app import get_git_commit

training_history_path = REPO_ROOT / "outputs" / "metrics" / "training_history.json"
history = {}
if training_history_path.exists():
    with open(training_history_path) as f:
        history = json.load(f)

manifest = {
    "phase": "06_Training",
    "training_history": history,
    "git_commit": get_git_commit(),
}
save_yaml(manifest, REPO_ROOT / "outputs" / "manifests" / "phase_06_training.yaml")
print("YAML manifest saved successfully:")
print(manifest)
